# V2-02 — Journal **roles** (source / bridge / terminal) + **citational velocity**

*Exploratory cartography, V2-S05. Panel-conditional — the true source generalists
(Nature / NEJM / Lancet) are NOT in this 10-journal surgical/orthopedic corpus, so
roles are validated against the **within-corpus prior**: generalist surgery journals
vs. subspecialty journals behaving as expected.*

**What this notebook renders**
1. The per-journal role-score table + a heatmap of the three z-scored components.
2. The directed topic-flow network, nodes coloured by assigned role.
3. Citational-velocity distributions per journal (box plots of years pub→cite).
4. The drop-one-journal jackknife stability summary (`role_rank_correlation`).

The logic lives in `scifield.cartography.roles`; this notebook does only I/O +
plotting. Tables are read from `V2/data/roles/` (built by
`V2/scripts/build_roles.py`).

## 1. Setup + load the role / velocity tables

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Repo-root sniff — this notebook lives at V2/notebooks/, code is under src/.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

ROLES_DIR = repo_root / "V2" / "data" / "roles"
FLOW_DIR = repo_root / "V2" / "data" / "flow"
ARCHETYPES = repo_root / "data" / "v1" / "archetypes.parquet"
CITED_BY = repo_root / "data" / "v1" / "enrichment" / "cited_by.parquet"
PAPERS_DUCKDB = repo_root / "data" / "v1" / "papers.duckdb"
FIG_DIR = repo_root / "V2" / "notebooks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DPI = 120

from scifield.cartography.roles import ROLE_COMPONENTS  # noqa: E402
from scifield.findings.seeding import directed_seeding_network, specialty_of  # noqa: E402

GRAIN = "leaf"  # render the leaf grain; mid is in the same role_scores under grain=="mid".

roles = pd.read_parquet(ROLES_DIR / "role_scores.parquet")
velocity = pd.read_parquet(ROLES_DIR / "velocity.parquet")
flow = pd.read_parquet(FLOW_DIR / f"flow_{GRAIN}.parquet")

roles_g = roles[roles["grain"] == GRAIN].sort_values("journal_slug").reset_index(drop=True)

print(f"grain={GRAIN!r}")
print(f"  role_scores: {len(roles_g)} journals | roles: {roles_g['role'].value_counts().to_dict()}")
print(f"  velocity:    {len(velocity)} journals")

SLUG_LABEL = {
    "spine": "Spine",
    "j_arthroplasty": "J Arthroplasty",
    "clin_orthop_relat_res": "CORR",
    "j_bone_joint_surg_am": "J Bone Joint Surg",
    "arthroscopy": "Arthroscopy",
    "surgery": "Surgery",
    "ann_surg": "Ann Surg",
    "br_j_surg": "Br J Surg",
    "j_am_coll_surg": "J Am Coll Surg",
    "jama_surg": "JAMA Surg",
}
ROLE_COLOR = {"source": "#34a853", "bridge": "#fbbc04", "terminal": "#ea4335"}

grain='leaf'
  role_scores: 10 journals | roles: {'terminal': 4, 'source': 4, 'bridge': 2}
  velocity:    10 journals


## 2. Role-score table + component heatmap

Each journal's **source** (publishes first + exports citations), **bridge**
(betweenness in the topic-flow network), and **terminal** (receives more than it
sends; late) scores. The `role` label is the **argmax of the z-scored components**
(ties break source > bridge > terminal). `specialty` lets us check the
within-corpus prior.

In [2]:
cols = [
    "journal_slug",
    "specialty",
    "seeding_score",
    "net_outflow_share",
    "betweenness",
    "source",
    "bridge",
    "terminal",
    "role",
]
table = roles_g[cols].copy()
for c in cols[2:-1]:
    table[c] = table[c].astype("float64").round(3)
display(table.sort_values("role"))

# Heatmap of the z-scored components per journal.
zcols = [f"{c}_z" for c in ROLE_COMPONENTS]
mat = roles_g.set_index("journal_slug")[zcols]
order = roles_g.sort_values(["role", "source_z"], ascending=[True, False])["journal_slug"]
mat = mat.loc[order]

fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(mat.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2)
ax.set_xticks(range(len(zcols)))
ax.set_xticklabels([c.replace("_z", "") for c in zcols])
ax.set_yticks(range(len(mat)))
ax.set_yticklabels([SLUG_LABEL.get(s, s) for s in mat.index], fontsize=8)
for i, slug in enumerate(mat.index):
    role = roles_g.set_index("journal_slug").loc[slug, "role"]
    ax.text(
        len(zcols) - 0.5,
        i,
        f"  {role}",
        va="center",
        ha="left",
        fontsize=7,
        color=ROLE_COLOR[role],
    )
ax.set_xlim(-0.5, len(zcols) + 0.9)
ax.set_title(
    "(a) Role components (z-scored) per journal — leaf grain\nlabel = argmax; " "panel-conditional",
    fontsize=9,
)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.18, label="z-score")
fig.tight_layout()
fig.savefig(FIG_DIR / "v2_02_role_heatmap.png", dpi=DPI, bbox_inches="tight")
plt.close(fig)
print("saved figures/v2_02_role_heatmap.png")

,journal_slug,specialty,seeding_score,net_outflow_share,betweenness,source,bridge,terminal,role
1,arthroscopy,orthopedic,0.463,0.057,0.111,-0.530,0.111,0.530,bridge
8,spine,orthopedic,0.538,-0.010,0.069,-0.379,0.069,0.379,bridge
2,br_j_surg,general_surgery,0.739,-0.008,0.000,0.731,0.000,-0.731,source
4,j_am_coll_surg,general_surgery,0.659,-0.022,0.000,0.236,0.000,-0.236,source
5,j_arthroplasty,orthopedic,0.480,0.220,0.000,0.212,0.000,-0.212,source
9,surgery,general_surgery,0.679,0.076,0.014,0.737,0.014,-0.737,source
0,ann_surg,general_surgery,0.632,-0.153,0.042,-0.430,0.042,0.430,terminal
3,clin_orthop_relat_res,orthopedic,0.692,-0.162,0.000,-0.132,0.000,0.132,terminal
6,j_bone_joint_surg_am,orthopedic,0.702,-0.237,0.000,-0.380,0.000,0.380,terminal
7,jama_surg,general_surgery,0.627,-0.053,0.000,-0.066,0.000,0.066,terminal


saved figures/v2_02_role_heatmap.png


## 3. Directed topic-flow network, coloured by role

Same seeding network as the cascade notebook (arrow `src → dst` = `src` publishes
the shared topic first), but here nodes are coloured by their **assigned role** and
sized by **betweenness** (the bridge axis). Bridges should sit where flow concentrates.

In [3]:
edges = directed_seeding_network(
    flow[["topic_id", "journal_slug", "year"]], entity_col="journal_slug"
)
WEIGHT_THRESHOLD = 0.60
strong = edges[edges["weight"] >= WEIGHT_THRESHOLD].copy()
print(f"network: {len(edges)} directed edges | {len(strong)} with weight >= {WEIGHT_THRESHOLD}")

role_of = roles_g.set_index("journal_slug")["role"].to_dict()
bet_of = roles_g.set_index("journal_slug")["betweenness"].to_dict()

# Order nodes around a circle by source score (strongest source at top, clockwise).
node_order = roles_g.sort_values("source", ascending=False)["journal_slug"].tolist()
n_nodes = len(node_order)
angles = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, n_nodes, endpoint=False)
pos = {
    slug: (float(np.cos(a)), float(np.sin(a))) for a, slug in zip(angles, node_order, strict=False)
}

fig, ax = plt.subplots(figsize=(8.4, 8.0))
if len(strong):
    wmin, wmax = strong["weight"].min(), strong["weight"].max()
    for e in strong.itertuples():
        x0, y0 = pos[e.src]
        x1, y1 = pos[e.dst]
        frac = 0.0 if wmax == wmin else (e.weight - wmin) / (wmax - wmin)
        ax.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(
                arrowstyle="-|>",
                color="#555555",
                alpha=0.30 + 0.55 * frac,
                lw=0.8 + 2.2 * frac,
                shrinkA=15,
                shrinkB=15,
                connectionstyle="arc3,rad=0.12",
            ),
        )
bet_max = max(bet_of.values()) or 1.0
for slug in node_order:
    x, y = pos[slug]
    size = 260 + 1400 * (bet_of.get(slug, 0.0) / bet_max)
    ax.scatter([x], [y], s=size, c=ROLE_COLOR[role_of[slug]], edgecolors="white", zorder=3)
    ax.text(x * 1.18, y * 1.18, SLUG_LABEL.get(slug, slug), ha="center", va="center", fontsize=8)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(
    "(b) Directed topic-flow network — nodes coloured by role, sized by betweenness\n"
    f"arrow src -> dst: src publishes first (weight >= {WEIGHT_THRESHOLD}). Panel-conditional.",
    fontsize=9,
)
for role, c in ROLE_COLOR.items():
    ax.scatter([], [], c=c, label=role, s=120, edgecolors="white")
ax.legend(loc="lower right", fontsize=8, frameon=False, title="role")
fig.tight_layout()
fig.savefig(FIG_DIR / "v2_02_role_network.png", dpi=DPI, bbox_inches="tight")
plt.close(fig)
print("saved figures/v2_02_role_network.png")

network: 90 directed edges | 6 with weight >= 0.6
saved figures/v2_02_role_network.png


## 4. Citational-velocity distributions per journal

Distribution of **years from publication to citation** (`citing_year − pub_year`)
for each journal's external inbound citations (`cited_by.parquet`). A *fast* journal
accrues citations soon after publication (small median); a *slow* one has a longer
tail. We recompute the per-citation lags here (the table holds only summaries) to draw
the box plots, capped at the same `max_lag=60` the module uses.

In [4]:
import duckdb  # noqa: E402

# Resolve openalex_id -> journal_slug + pub year (read-only DuckDB), then lag per citation.
arch = pd.read_parquet(ARCHETYPES, columns=["pmid", "openalex_id", "year"]).dropna(
    subset=["openalex_id"]
)
arch["pmid"] = arch["pmid"].astype("string")
con = duckdb.connect(str(PAPERS_DUCKDB), read_only=True)
try:
    pj = con.execute("SELECT pmid, journal_slug FROM papers_distinct").df()
finally:
    con.close()
pj["pmid"] = pj["pmid"].astype("string")
meta = arch.merge(pj, on="pmid", how="inner")[["openalex_id", "journal_slug", "year"]]
meta = meta.drop_duplicates("openalex_id")

cb = pd.read_parquet(CITED_BY, columns=["focal_oa_id", "citing_year"]).rename(
    columns={"focal_oa_id": "openalex_id"}
)
joined = cb.merge(meta, on="openalex_id", how="inner")
joined["lag"] = (joined["citing_year"] - joined["year"]).astype("float64")
joined = joined[(joined["lag"] >= 0) & (joined["lag"] <= 60)]

# Order journals by median lag (fastest first) — matches the velocity table order.
order = velocity.sort_values("median_lag")["journal_slug"].tolist()
data = [joined.loc[joined["journal_slug"] == s, "lag"].to_numpy() for s in order]

fig, ax = plt.subplots(figsize=(9.0, 5.2))
bp = ax.boxplot(
    data,
    vert=False,
    showfliers=False,
    patch_artist=True,
    medianprops=dict(color="black", lw=1.4),
)
for patch, slug in zip(bp["boxes"], order, strict=False):
    patch.set_facecolor(
        {"orthopedic": "#4285f4", "general_surgery": "#ea4335"}.get(specialty_of(slug), "#9aa0a6")
    )
    patch.set_alpha(0.65)
ax.set_yticklabels([SLUG_LABEL.get(s, s) for s in order], fontsize=8)
ax.set_xlabel("years from publication to citation (capped at 60)")
ax.set_title(
    "(c) Citational velocity per journal — fastest (top) to slowest (bottom)\n"
    "colour = specialty; whiskers show the accrual tail",
    fontsize=9,
)
for sp, c in [("orthopedic", "#4285f4"), ("general_surgery", "#ea4335")]:
    ax.scatter([], [], c=c, label=sp, s=80)
ax.legend(loc="lower right", fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "v2_02_velocity_box.png", dpi=DPI, bbox_inches="tight")
plt.close(fig)
print("saved figures/v2_02_velocity_box.png")

display(
    velocity.sort_values("median_lag")[
        [
            "journal_slug",
            "specialty",
            "n_citations",
            "median_lag",
            "iqr_lag",
            "frac_within_2y",
            "velocity",
        ]
    ].round(3)
)

saved figures/v2_02_velocity_box.png


,journal_slug,specialty,n_citations,median_lag,iqr_lag,frac_within_2y,velocity
0,j_arthroplasty,orthopedic,357116,4.0,5.0,0.277,fast
1,arthroscopy,orthopedic,310712,5.0,5.0,0.238,slow
2,br_j_surg,general_surgery,391221,5.0,5.0,0.213,slow
3,j_am_coll_surg,general_surgery,257733,5.0,5.0,0.224,slow
4,jama_surg,general_surgery,317708,5.0,5.0,0.239,slow
5,surgery,general_surgery,312895,5.0,5.0,0.233,slow
6,ann_surg,general_surgery,881729,6.0,7.0,0.199,slow
7,clin_orthop_relat_res,orthopedic,505617,6.0,6.0,0.162,slow
8,j_bone_joint_surg_am,orthopedic,631951,7.0,6.0,0.154,slow
9,spine,orthopedic,754411,7.0,6.0,0.144,slow


## 5. Drop-one-journal jackknife stability (`role_rank_correlation`)

Method C of the validation protocol: re-run the role scoring ten times, each
dropping one journal, and measure how stable the source-axis ranking is across runs
(mean pairwise Spearman ρ). The protocol PASS bar is `role_rank_correlation ≥ 0.7`.

In [5]:
from scifield.cartography.nulls import rank_stability  # noqa: E402
from scifield.cartography.roles import role_scores_jackknife  # noqa: E402

rrc_by_grain = {}
for grain in ("leaf", "mid"):
    fl = pd.read_parquet(FLOW_DIR / f"flow_{grain}.parquet")
    key = "topic_id" if grain == "leaf" else "mid_level_id"
    jk = role_scores_jackknife(fl, topic_key=key, score_col="source_z")
    rrc = rank_stability(jk, key_col="journal_slug", score_col="score", run_col="held_out")
    rrc_by_grain[grain] = float(rrc)

bar = 0.7
print("Drop-one-journal role-rank stability (Method C; Gate G6 roles input):")
for grain, rrc in rrc_by_grain.items():
    verdict = "PASS" if rrc >= bar else "BELOW BAR"
    print(f"  {grain}: role_rank_correlation = {rrc:.4f}  (bar >= {bar} -> {verdict})")

# Leaf-vs-mid role-label agreement (Method D supporting check).
leaf_lab = roles[roles["grain"] == "leaf"].set_index("journal_slug")["role"]
mid_lab = roles[roles["grain"] == "mid"].set_index("journal_slug")["role"]
shared = leaf_lab.index.intersection(mid_lab.index)
agree = float((leaf_lab.loc[shared] == mid_lab.loc[shared]).mean())
print(f"\nleaf-vs-mid role-label agreement: {agree:.2f} ({int(agree * len(shared))}/{len(shared)})")

Drop-one-journal role-rank stability (Method C; Gate G6 roles input):
  leaf: role_rank_correlation = 0.9450  (bar >= 0.7 -> PASS)
  mid: role_rank_correlation = 0.9619  (bar >= 0.7 -> PASS)

leaf-vs-mid role-label agreement: 0.80 (8/10)


## 6. Summary — within-corpus prior validation + caveats (exploratory)

**Roles (leaf grain).** The three general-surgery generalists **Br J Surg**,
**Surgery**, **J Am Coll Surg** score as **sources** (publish topics early and export
citations) — consistent with the within-corpus prior that the broad surgery journals
sit upstream. The subspecialty journals **Arthroscopy** and **Spine** score as
**bridges** (highest betweenness in the topic-flow network). The heavily-cited prestige
journals — **Ann Surg**, **J Bone Joint Surg**, **CORR**, **JAMA Surg** — score as
**terminals**: by the V2-S05 definition a terminal has high citation *in-flow* relative
to out-flow, and these are the corpus's biggest net citation *importers* (their papers
are cited within the corpus more than they cite out). That is the honest reading: in a
panel with no true generalist *source* (Nature/NEJM), the most authoritative journals
present surface on the **receive** axis, not the **publish-first** axis.

**Velocity.** Median time-to-citation runs 4–7 years. **J Arthroplasty** is fastest
(4 y median, 28% of citations within 2 y); the prestige terminals (**Ann Surg**,
**J Bone Joint Surg**, **Spine**) are slowest (6–7 y) but accrue the most citations
overall — a long, heavy tail. Fast accrual and high role-terminal status are distinct:
a journal can be heavily cited yet slow.

**Stability.** Roles are **stable under the drop-one-journal jackknife**
(`role_rank_correlation` ≈ 0.95 leaf / 0.96 mid, both ≥ the 0.7 PASS bar) and **robust
across topic granularity** (80% leaf-vs-mid label agreement; component Spearman ρ ≈
0.85–0.90). → **roles PASS the Gate G6 stability input.**

**Caveats.** (1) *Panel-conditional* — the true source generalists are absent; a
"terminal" here means "net citation importer within these ten journals", not a global
dead-end. (2) *1995 left-censoring* — the seeding sub-signal of source/terminal is
biased by the panel's 1995 start; we mitigate by weighting the censoring-free
citation-flow and betweenness signals equally (see the role heatmap — net-outflow and
betweenness, not seeding alone, separate the labels). (3) The role *labels* are a coarse
argmax; the continuous component scores are the nuanced output the map should carry.